In [1]:
import numpy as np, cv2, random
from PIL import Image
import os

os.makedirs('../data/digits/empty', exist_ok=True)

def make_realistic(img):
    a = np.array(img, dtype=np.float32)

    # random contrast/brightness (gray, uneven backgrounds)
    a = a * random.uniform(0.6, 1.0) + random.uniform(0, 40)

    # blur
    if random.random() < 0.5:
        k = random.choice([3, 5])
        a = cv2.GaussianBlur(a, (k, k), 0)

    # grid-line fragments on 1-2 edges
    for _ in range(random.randint(0, 2)):
        t = random.randint(1, 2)
        side = random.choice(['top', 'bottom', 'left', 'right'])
        v = random.randint(0, 80)
        if side == 'top':    a[:t, :] = v
        if side == 'bottom': a[-t:, :] = v
        if side == 'left':   a[:, :t] = v
        if side == 'right':  a[:, -t:] = v

    # noise
    a = a + np.random.normal(0, random.uniform(2, 10), a.shape)
    return np.clip(a, 0, 255).astype(np.uint8)

def generate_empty_cell(size=28):
    base = np.random.randint(230, 255, (size, size), dtype=np.uint8)
    noise = np.random.normal(0, 5, (size, size))
    img = np.clip(base + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(img, mode='L')

for i in range(400):
    img = generate_empty_cell(size=28)
    img = Image.fromarray(make_realistic(img))      # new line
    img.save(f'../data/digits/empty/empty_{i}.png')

print("Done:", len(os.listdir('../data/digits/empty')))

Done: 400


In [2]:
from PIL import Image, ImageDraw, ImageFont
import os, random

fonts_dir = 'C:/Windows/Fonts'
font_files = [f for f in os.listdir(fonts_dir) if f.endswith('.ttf')][:15]

def generate_digit_image(digit, font_path, size=28):
    img = Image.new('L', (size, size), color=255)
    draw = ImageDraw.Draw(img)
    font_size = random.randint(18, 24)
    font = ImageFont.truetype(font_path, font_size)
    bbox = draw.textbbox((0,0), str(digit), font=font)
    w, h = bbox[2]-bbox[0], bbox[3]-bbox[1]
    x = (size - w) / 2 - bbox[0]
    y = (size - h) / 2 - bbox[1]
    draw.text((x, y), str(digit), fill=0, font=font)

   
    return img

for digit in range(1, 10):
    os.makedirs(f'../data/digits/{digit}', exist_ok=True)
    for i, font_file in enumerate(font_files):
        for variant in range(20):  # 20 variants per font
            img = generate_digit_image(digit, f'{fonts_dir}/{font_file}')
            img = img.rotate(random.uniform(-10, 10), fillcolor=255)
            img.save(f'../data/digits/{digit}/synth_{font_file}_{i}_{variant}.png')

print("Synthetic generation done.")

Synthetic generation done.


In [3]:
import os

digits_path = '../data/digits'
for cls in sorted(os.listdir(digits_path)):
    cls_path = os.path.join(digits_path, cls)
    if os.path.isdir(cls_path):
        count = len(os.listdir(cls_path))
        print(f"{cls}: {count} images")

1: 400 images
2: 400 images
3: 400 images
4: 400 images
5: 400 images
6: 400 images
7: 400 images
8: 400 images
9: 400 images
empty: 400 images
